In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import math
from scipy.signal import butter, sosfiltfilt

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import Lasso
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
import scipy
from pathlib import Path

In [3]:
import matplotlib.pyplot as plt

In [4]:
# s1_data_proc = emg_proc(s1_data)

# plt.figure(figsize=(8,2))
# plt.imshow(np.log(np.abs(s1_data[0,:,:].T)), aspect='auto')
# plt.figure(figsize=(8,2))
# plt.imshow(np.log(np.abs(s1_data_proc[0,:,:].T)), aspect='auto')

In [5]:
# print(s1_data_proc[:,::100,:].shape,
# np.concatenate(s1_data_proc[:,::100,:].T, 0).T.shape)

In [6]:
# %matplotlib qt5
# def plot_feat_data():
#     images = list(s1_data_proc[:,::100,:])
#     N = len(images)
#     if N == 0:
#         raise ValueError("Список images пуст.")
#     ncols = None
#     if ncols is None:
#         ncols = math.ceil(math.sqrt(N))
#     nrows = math.ceil(N / ncols)

#     vmin=None
#     vmax=None

    
#     fig, axes = plt.subplots(nrows, ncols, figsize=(12, 12), squeeze=True)
#     k = 0
#     for r in range(nrows):
#         for c in range(ncols):
#             ax = axes[r, c]
#             if k < N:
#                 ax.imshow(images[k], interpolation=None, aspect='auto', vmin=vmin, vmax=vmax)
#                 ax.set_xticks([])
#                 ax.set_yticks([])
#                 ax.set_title(f'{labels[k]}')
#             else:
#                 ax.axis("off")
#             k += 1
#     fig.tight_layout()


#     # плотнее расположим мини-графики
#     plt.subplots_adjust(wspace=0.05, hspace=0.05)

In [7]:
# def plot_feat_mean():
#     mean_emg = []
#     for i in range(10):
#         temp = s1_data_proc[:,::100,:][s1_labels==i]
#         mean_emg.append(temp.mean(axis=0))
#     mean_emg = np.array(mean_emg)

#     fig, axes = plt.subplots(3, 4, figsize=(4, 4), squeeze=True)
#     k = 0
#     for r in range(3):
#         for c in range(4):
#             ax = axes[r, c]
#             if k < len(mean_emg):
#                 ax.imshow(mean_emg[k], interpolation=None, aspect='auto')
#                 ax.set_xticks([])
#                 ax.set_yticks([])
#             else:
#                 ax.axis("off")
#             k += 1
#     fig.tight_layout()

In [8]:
# Define Transformer Model with increased dropout and reduced complexity
class EMGTransformer(nn.Module):
    def __init__(self, input_dim, n_classes, n_heads=2, ff_dim=64, conv_out_dim=32, num_layers=2, dropout=0.4):
        super(EMGTransformer, self).__init__()
        self.input_layer = nn.Linear(input_dim, ff_dim)
        # self.conv_layer = nn.Sequential(
        #     nn.Conv1d(ff_dim, 128, kernel_size=4, stride=3, padding=1),
        #     nn.Conv1d(128, 64, kernel_size=3, stride=2, padding=1),
        #     nn.Conv1d(64, conv_out_dim, kernel_size=2, stride=1, padding=1))


        encoder_layers = nn.TransformerEncoderLayer(
            d_model=ff_dim, nhead=n_heads, dim_feedforward=ff_dim, dropout=dropout
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        
        self.dropout = nn.Dropout(p=dropout)
        self.classifier = nn.Sequential(
            nn.Linear(ff_dim, 32),
            nn.ReLU(),
            self.dropout,
            nn.Linear(32, n_classes)
        )
        
    def forward(self, x):
        x = self.input_layer(x)
        x = x.unsqueeze(1)  # Add a sequence dimension (required for Transformer)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)  # Global average pooling over the sequence dimension
        x = self.classifier(x)
        return x

In [9]:
# Validation function to monitor validation loss for early stopping
def validate_model(model, val_loader, criterion):
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
    return val_loss / len(val_loader)

In [10]:
# Training function with early stopping
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=50, patience=5):
    model.train()
    best_val_loss = float('inf')
    patience_counter = 0
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        # Early stopping based on validation loss
        val_loss = validate_model(model, val_loader, criterion)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0  # Reset patience counter if validation loss improves
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break
        
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}, Val Loss: {val_loss}")

In [11]:
# Evaluation function
def evaluate_model(model, test_loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return np.array(all_preds), np.array(all_labels)

In [14]:
data_path = Path(r"C:\Users\yurki\Downloads\dataset.mat").as_posix()

data= scipy.io.loadmat(data_path)
labels = np.squeeze(data['label'])
sub_marks = np.load('sub_marks.npy')

print(len(data['data']),
len(labels),
len(sub_marks))

s1_data = data['data'][sub_marks==0]
print(s1_data.shape)

s1_labels = labels[sub_marks==0]

5530 5530 5530
(498, 5600, 8)


In [15]:
def emg_proc(emg, lp=5, hp=40, fs=1000):
    sos_hp = butter(2, hp/(fs/2), btype='highpass', output='sos')
    emg_hp = sosfiltfilt(sos_hp, emg, axis=1)

    emg_abs = np.abs(emg_hp)
    sos_lp = butter(2, lp/(fs/2), btype='lowpass', output='sos')
    env = sosfiltfilt(sos_lp, emg_abs, axis=1)
    return env

In [16]:
def save_feat_data():
    data_proc = emg_proc(data['data'])
    data_feat = np.concatenate(data_proc[:,::100,:].T, 0).T
    print(len(data_feat), len(labels), len(sub_marks))
    # scipy.io.savemat('feat_dataset.mat', {'data': data_feat, 'label': labels, 'sub': sub_marks})
# save_feat_data()

In [17]:
data_proc = emg_proc(data['data'])

In [18]:
emg = np.concatenate(data_proc[:,::40,:].T, 0).T
print(emg.shape, len(labels), len(sub_marks))

(5530, 1120) 5530 5530


In [19]:
# Hyperparameters
#Changed X_scaled to X since we are not using feature selection anymore
input_dim = emg.shape[1] 
n_classes = 10  # Number of classes
n_heads = 2  # Reduced number of heads
num_layers = 5
ff_dim = 128  # Reduced feedforward dimension
conv_out_dim = 128
dropout = 0.15  # Increased dropout
learning_rate = 1e-4
epochs = 150
k_folds = 6
weight_decay = 1e-3  # Increased L2 regularization

# Early stopping parameters
early_stopping_patience = 5

In [24]:
# Initialize the model, loss function, and optimizer with weight decay
model = EMGTransformer(input_dim=input_dim, n_classes=n_classes, n_heads=n_heads, ff_dim=ff_dim, conv_out_dim=32, num_layers=2, dropout=dropout)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

C:\Users\yurki\AppData\Local\Temp\ipykernel_16564\3302714385.py:15: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)


In [25]:
# K-Fold Cross-Validation
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

In [26]:
sub = 10
X = emg[sub_marks==sub]
y = labels[sub_marks==sub]
print(X.shape, len(y))
print(np.unique(y))

(548, 1120) 548
[0 1 2 3 4 5 6 7 8 9]


In [27]:
# Convert to PyTorch tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)
print(X_tensor.shape, y_tensor.shape)

torch.Size([548, 1120]) torch.Size([548])


In [28]:
# Metrics storage
fold_metrics = {
    'accuracy': [],
    'precision': [],
    'recall': [],
    'f1': [],
    'confusion_matrices': []
}

# Perform K-Fold Cross Validation
for fold, (train_index, test_index) in enumerate(kf.split(X_tensor)):
    print(f'Fold {fold + 1}/{k_folds}')
    
    # Split data into training and test sets for this fold
    X_train, X_test = X_tensor[train_index], X_tensor[test_index]
    y_train, y_test = y_tensor[train_index], y_tensor[test_index]
    
    # Further split training data into train and validation sets for early stopping
    train_idx, val_idx = train_test_split(np.arange(len(X_train)), test_size=0.2, random_state=42)
    X_train_fold, X_val_fold = X_train[train_idx], X_train[val_idx]
    y_train_fold, y_val_fold = y_train[train_idx], y_train[val_idx]
    
    # Create DataLoader for batching
    train_dataset = TensorDataset(X_train_fold, y_train_fold)
    val_dataset = TensorDataset(X_val_fold, y_val_fold)
    test_dataset = TensorDataset(X_test, y_test)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    
    # Reset model, criterion, and optimizer for each fold
    model = EMGTransformer(input_dim=input_dim, n_classes=n_classes,
                           n_heads=n_heads, ff_dim=ff_dim,
                        #    conv_out_dim=32,
                           num_layers=num_layers, dropout=dropout)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    # Train the model with early stopping
    train_model(model, train_loader, val_loader,
                criterion, optimizer, epochs=epochs,
                patience=early_stopping_patience)
    
    # Evaluate the model
    y_pred, y_true = evaluate_model(model, test_loader)

    # Calculate metrics
    fold_accuracy = np.mean(y_pred == y_true)
    fold_precision = precision_score(y_true, y_pred, average='weighted')
    fold_recall = recall_score(y_true, y_pred, average='weighted')
    fold_f1 = f1_score(y_true, y_pred, average='weighted')
    fold_confusion_matrix = confusion_matrix(y_true, y_pred)
    
    fold_metrics['accuracy'].append(fold_accuracy)
    fold_metrics['precision'].append(fold_precision)
    fold_metrics['recall'].append(fold_recall)
    fold_metrics['f1'].append(fold_f1)
    fold_metrics['confusion_matrices'].append(fold_confusion_matrix)
    
    print(f'Fold {fold + 1} Accuracy: {fold_accuracy * 100:.2f}%')
    print(f'Fold {fold + 1} Precision: {fold_precision:.2f}')
    print(f'Fold {fold + 1} Recall: {fold_recall:.2f}')
    print(f'Fold {fold + 1} F1 Score: {fold_f1:.2f}')
    print(f'Fold {fold + 1} Confusion Matrix:\n{fold_confusion_matrix}')

Fold 1/6


C:\Users\yurki\AppData\Local\Temp\ipykernel_16564\3302714385.py:15: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)


Epoch 1/150, Loss: 2.3191125790278115, Val Loss: 2.335259437561035
Epoch 2/150, Loss: 2.3004552125930786, Val Loss: 2.328017473220825
Epoch 3/150, Loss: 2.283105472723643, Val Loss: 2.3077282110850015
Epoch 4/150, Loss: 2.271714150905609, Val Loss: 2.2858522733052573
Epoch 5/150, Loss: 2.2361263632774353, Val Loss: 2.2337217330932617
Epoch 6/150, Loss: 2.2184266646703086, Val Loss: 2.205599864323934
Epoch 7/150, Loss: 2.098409910996755, Val Loss: 2.0831775665283203
Epoch 8/150, Loss: 1.935566822687785, Val Loss: 1.9013756116231282
Epoch 9/150, Loss: 1.7296165227890015, Val Loss: 1.6937796672185261
Epoch 10/150, Loss: 1.5446566939353943, Val Loss: 1.6433669328689575
Epoch 11/150, Loss: 1.4303913911183674, Val Loss: 1.4872780243555705
Epoch 12/150, Loss: 1.2848361829916637, Val Loss: 1.3437258799870808
Epoch 13/150, Loss: 1.1522240738073986, Val Loss: 1.2311698198318481
Epoch 14/150, Loss: 1.058687354127566, Val Loss: 1.215306282043457
Epoch 15/150, Loss: 0.9745234449704488, Val Loss: 1.

C:\Users\yurki\AppData\Local\Temp\ipykernel_16564\3302714385.py:15: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)


Epoch 1/150, Loss: 2.345209777355194, Val Loss: 2.3433109124501548
Epoch 2/150, Loss: 2.3035998145739236, Val Loss: 2.2910746733347573
Epoch 3/150, Loss: 2.282754679520925, Val Loss: 2.315231720606486
Epoch 4/150, Loss: 2.2572320898373923, Val Loss: 2.270442088445028
Epoch 5/150, Loss: 2.207786520322164, Val Loss: 2.1633352438608804
Epoch 6/150, Loss: 2.023640990257263, Val Loss: 1.9465211629867554
Epoch 7/150, Loss: 1.8004693190256755, Val Loss: 1.7565324703852336
Epoch 8/150, Loss: 1.6352335313955944, Val Loss: 1.595617651939392
Epoch 9/150, Loss: 1.469644953807195, Val Loss: 1.4840209484100342
Epoch 10/150, Loss: 1.3518327474594116, Val Loss: 1.3673341671625774
Epoch 11/150, Loss: 1.208845282594363, Val Loss: 1.2353386878967285
Epoch 12/150, Loss: 1.1107176691293716, Val Loss: 1.1415690183639526
Epoch 13/150, Loss: 1.0476462294658024, Val Loss: 1.0904930432637532
Epoch 14/150, Loss: 0.9404507776101431, Val Loss: 1.0287038485209148
Epoch 15/150, Loss: 0.8789853801329931, Val Loss: 0.

...

Epoch 1/150, Loss: 2.326139271259308, Val Loss: 2.3005358378092446
Epoch 2/150, Loss: 2.306814273198446, Val Loss: 2.2759454250335693
Epoch 3/150, Loss: 2.2844165364901223, Val Loss: 2.269167900085449
Epoch 4/150, Loss: 2.2506878773371377, Val Loss: 2.17983341217041
Epoch 5/150, Loss: 2.1310853560765586, Val Loss: 1.9659424622853596
Epoch 6/150, Loss: 1.9412008623282115, Val Loss: 1.8336491584777832
Epoch 7/150, Loss: 1.7825364470481873, Val Loss: 1.7230724891026814
Epoch 8/150, Loss: 1.6645484069983165, Val Loss: 1.5598338047663372
Epoch 9/150, Loss: 1.5104213158289592, Val Loss: 1.4484587907791138
Epoch 10/150, Loss: 1.371059000492096, Val Loss: 1.349103331565857
Epoch 11/150, Loss: 1.2598060468832653, Val Loss: 1.2341545820236206
Epoch 12/150, Loss: 1.1594462593396504, Val Loss: 1.1432488759358723
Epoch 13/150, Loss: 1.056413173675537, Val Loss: 1.0550953149795532
Epoch 14/150, Loss: 1.0004635999600093, Val Loss: 0.9887348810831705
Epoch 15/150, Loss: 0.9209324419498444, Val Loss: 0

C:\Users\yurki\AppData\Local\Temp\ipykernel_16564\3302714385.py:15: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)


Epoch 1/150, Loss: 2.308887243270874, Val Loss: 2.2967403729756675
Epoch 2/150, Loss: 2.2800705830256143, Val Loss: 2.2553796768188477
Epoch 3/150, Loss: 2.2374998132387796, Val Loss: 2.2075424194335938
Epoch 4/150, Loss: 2.1222404837608337, Val Loss: 2.0558667182922363
Epoch 5/150, Loss: 2.0134145816167197, Val Loss: 1.902721365292867
Epoch 6/150, Loss: 1.847563515106837, Val Loss: 1.7715219259262085
Epoch 7/150, Loss: 1.7226949135462444, Val Loss: 1.6396656433741252
Epoch 8/150, Loss: 1.6190739671389263, Val Loss: 1.556766112645467
Epoch 9/150, Loss: 1.51617032289505, Val Loss: 1.4795012474060059
Epoch 10/150, Loss: 1.4048970639705658, Val Loss: 1.3590336640675862
Epoch 11/150, Loss: 1.3159119784832, Val Loss: 1.2515325943628948
Epoch 12/150, Loss: 1.214434524377187, Val Loss: 1.2028474807739258
Epoch 13/150, Loss: 1.1068149656057358, Val Loss: 1.0975993474324544
Epoch 14/150, Loss: 1.0272271086772282, Val Loss: 1.0035037597020466
Epoch 15/150, Loss: 0.9215002208948135, Val Loss: 0.9

C:\Users\yurki\AppData\Local\Temp\ipykernel_16564\3302714385.py:15: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)


Epoch 1/150, Loss: 2.3000988960266113, Val Loss: 2.30721378326416
Epoch 2/150, Loss: 2.2574369708697, Val Loss: 2.265806039174398
Epoch 3/150, Loss: 2.14292178551356, Val Loss: 2.098086675008138
Epoch 4/150, Loss: 1.9969562689463298, Val Loss: 1.9548956950505574
Epoch 5/150, Loss: 1.8393311897913616, Val Loss: 1.8634703159332275
Epoch 6/150, Loss: 1.7079045176506042, Val Loss: 1.678541938463847
Epoch 7/150, Loss: 1.5815363824367523, Val Loss: 1.590875506401062
Epoch 8/150, Loss: 1.4654055734475453, Val Loss: 1.5009679396947224
Epoch 9/150, Loss: 1.3822010060151417, Val Loss: 1.3924401601155598
Epoch 10/150, Loss: 1.283500333627065, Val Loss: 1.3455528418223064
Epoch 11/150, Loss: 1.178800215323766, Val Loss: 1.2157373428344727
Epoch 12/150, Loss: 1.077744538585345, Val Loss: 1.1857223908106487
Epoch 13/150, Loss: 0.9785035302241644, Val Loss: 1.057257890701294
Epoch 14/150, Loss: 0.9046283562978109, Val Loss: 0.9589182138442993
Epoch 15/150, Loss: 0.8297486553589503, Val Loss: 0.894382

In [29]:
fold_metrics['accuracy']

[np.float64(0.9130434782608695),
 np.float64(0.9456521739130435),
 np.float64(0.967032967032967),
 np.float64(0.967032967032967),
 np.float64(0.945054945054945),
 np.float64(0.978021978021978)]

In [30]:
# Print overall average metrics across all folds
avg_accuracy = np.mean(fold_metrics['accuracy'])
avg_precision = np.mean(fold_metrics['precision'])
avg_recall = np.mean(fold_metrics['recall'])
avg_f1 = np.mean(fold_metrics['f1'])

print(f'\n- Average Accuracy across {k_folds} folds: {avg_accuracy * 100:.2f}%')
print(f'- Average Precision across {k_folds} folds: {avg_precision:.2f}')
print(f'- Average Recall across {k_folds} folds: {avg_recall:.2f}')
print(f'- Average F1 Score across {k_folds} folds: {avg_f1:.2f}')

avg_cm = np.mean(fold_metrics['confusion_matrices'], axis=0).round(2)
print('- ', avg_cm)


- Average Accuracy across 6 folds: 95.26%
- Average Precision across 6 folds: 0.96
- Average Recall across 6 folds: 0.95
- Average F1 Score across 6 folds: 0.95
-  [[8.67 0.   0.   0.   0.   0.17 0.17 0.   0.17 0.  ]
 [0.   8.67 0.   0.   0.33 0.   0.   0.   0.17 0.  ]
 [0.   0.   8.33 0.   0.17 0.33 0.   0.   0.17 0.17]
 [0.   0.   0.17 8.33 0.17 0.   0.   0.33 0.   0.  ]
 [0.   0.   0.   0.   8.67 0.17 0.   0.33 0.   0.  ]
 [0.   0.   0.33 0.   0.33 8.5  0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   9.   0.   0.   0.17]
 [0.   0.   0.   0.   0.17 0.   0.   9.   0.   0.  ]
 [0.   0.   0.   0.17 0.   0.   0.   0.   9.   0.  ]
 [0.   0.17 0.   0.   0.   0.   0.   0.   0.   8.83]]


**sub0**
- Average Accuracy across 6 folds: 83.94%
- Average Precision across 6 folds: 0.86
- Average Recall across 6 folds: 0.84
- Average F1 Score across 6 folds: 0.84
-  [[7.   0.   0.17 0.   0.   0.   0.33 0.   0.5  0.33]
 [0.67 7.33 0.   0.   0.   0.   0.33 0.   0.   0.  ]
 [0.   0.   7.5  0.   0.   0.5  0.   0.   0.17 0.  ]
 [0.33 0.17 0.   7.17 0.   0.17 0.   0.   0.   0.5 ]
 [0.33 0.   0.17 0.33 6.83 0.33 0.17 0.17 0.   0.  ]
 [0.17 0.   0.17 0.   0.5  6.5  0.17 0.   0.33 0.33]
 [0.17 0.67 0.   0.   0.   0.17 6.83 0.   0.33 0.17]
 [0.17 0.   0.33 0.17 0.33 0.   0.   7.   0.17 0.17]
 [1.17 0.   0.   0.   0.   0.33 0.17 0.   6.83 0.  ]
 [0.67 0.33 0.   0.17 0.17 0.   0.   0.17 0.   6.67]]
**sub1**
- Average Accuracy across 6 folds: 93.81%
- Average Precision across 6 folds: 0.95
- Average Recall across 6 folds: 0.94
- Average F1 Score across 6 folds: 0.94
-  [[8.83 0.   0.   0.   0.   0.   0.   0.   0.33 0.17]
 [0.17 8.33 0.17 0.   0.17 0.   0.   0.17 0.   0.  ]
 [0.17 0.   8.5  0.   0.   0.17 0.17 0.   0.17 0.  ]
 [0.   0.17 0.5  8.17 0.17 0.   0.   0.   0.17 0.  ]
 [0.   0.17 0.   0.17 8.5  0.17 0.   0.   0.   0.17]
 [0.   0.   0.   0.   0.17 8.83 0.17 0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   9.   0.17 0.   0.  ]
 [0.   0.   0.   0.17 0.17 0.17 0.17 8.33 0.17 0.  ]
 [0.   0.   0.   0.   0.   0.17 0.33 0.   8.67 0.  ]
 [0.   0.   0.   0.   0.   0.17 0.   0.   0.17 8.67]]
**sub2**
- Average Accuracy across 6 folds: 72.17%
- Average Precision across 6 folds: 0.74
- Average Recall across 6 folds: 0.72
- Average F1 Score across 6 folds: 0.71
-  [[7.83 0.17 0.17 0.33 0.   0.   0.   0.   0.  ]
 [0.17 7.83 0.   0.33 0.   0.   0.   0.   0.  ]
 [0.17 1.   6.5  0.67 0.   0.   0.   0.   0.  ]
 [0.5  0.33 0.5  6.67 0.   0.   0.17 0.   0.  ]
 [0.   0.   0.   0.   5.   0.33 0.5  1.17 1.33]
 [0.   0.   0.   0.   0.83 6.33 0.17 0.83 0.17]
 [0.   0.   0.   0.17 0.5  0.33 7.   0.17 0.33]
 [0.   0.   0.   0.   2.17 1.17 0.   4.83 0.  ]
 [0.   0.   0.   0.   1.83 1.5  1.   1.83 2.  ]]
**sub3**
- Average Accuracy across 6 folds: 80.65%
- Average Precision across 6 folds: 0.82
- Average Recall across 6 folds: 0.81
- Average F1 Score across 6 folds: 0.80
-  [[7.5  0.   0.   0.   0.   0.   0.17 0.   0.67 0.  ]
 [0.   7.33 0.33 0.   0.17 0.   0.   0.   0.   0.33]
 [0.   0.17 6.83 0.33 0.33 0.5  0.   0.   0.   0.  ]
 [0.   0.   0.17 4.5  0.17 0.17 0.17 0.83 0.5  0.17]
 [0.   0.   0.5  0.17 6.5  0.5  0.   0.17 0.   0.  ]
 [0.   0.   0.17 0.67 0.5  5.5  0.   1.33 0.17 0.  ]
 [0.33 0.   0.17 0.33 0.   0.   6.5  0.   0.33 0.17]
 [0.   0.17 0.17 0.17 0.17 0.67 0.   6.33 0.17 0.  ]
 [0.33 0.   0.   0.   0.   0.17 1.   0.17 6.33 0.17]
 [0.   0.17 0.   0.17 0.5  0.17 0.17 0.17 0.   6.67]]
**sub4**

- Average Accuracy across 6 folds: 82.46%
- Average Precision across 6 folds: 0.85
- Average Recall across 6 folds: 0.82
- Average F1 Score across 6 folds: 0.83
-  [[7.17 0.5  0.   0.5  0.   0.   0.   0.   0.   0.17]
 [0.17 7.17 0.5  0.17 0.17 0.17 0.   0.17 0.   0.  ]
 [0.   0.   8.17 0.   0.   0.   0.17 0.   0.   0.  ]
 [0.67 0.17 0.33 6.33 0.17 0.   0.   0.   0.17 0.17]
 [0.   0.17 0.17 0.   6.   0.5  0.   0.33 0.   1.17]
 [0.17 0.   0.   0.   0.33 6.17 0.17 1.   0.   0.33]
 [0.   0.   0.17 0.   0.   0.33 7.   0.17 0.17 0.17]
 [0.   0.   0.   0.   0.   1.   0.33 5.5  0.67 0.17]
 [0.   0.   0.   0.17 0.   0.   0.   0.5  7.33 0.17]
 [0.   0.   0.   0.33 0.5  0.33 0.17 0.17 0.17 6.5 ]]
**sub5**
- Average Accuracy across 6 folds: 67.61%
- Average Precision across 6 folds: 0.71
- Average Recall across 6 folds: 0.68
- Average F1 Score across 6 folds: 0.67
-  [[5.83 0.17 0.   0.33 0.33 0.17 0.   0.   0.5  0.5 ]
 [0.17 4.   0.33 0.83 0.   1.83 0.83 0.17 0.   0.  ]
 [0.17 0.   4.83 0.83 0.33 0.5  0.   1.17 0.5  0.  ]
 [0.17 0.67 0.67 4.33 0.33 0.67 0.5  0.67 0.17 0.17]
 [0.17 0.   0.17 0.17 6.83 0.   0.   0.5  0.17 0.17]
 [0.33 0.5  0.33 0.5  0.17 6.17 0.   0.33 0.   0.  ]
 [0.33 0.5  0.33 0.5  0.   0.33 5.33 0.5  0.   0.17]
 [0.17 0.17 0.17 0.5  0.17 0.67 0.5  6.   0.   0.  ]
 [0.83 0.   0.17 0.17 0.   0.33 0.33 0.5  5.33 0.67]
 [0.5  0.   0.33 0.   0.   0.17 0.   0.   0.33 6.67]]
**sub6**
- Average Accuracy across 6 folds: 90.06%
- Average Precision across 6 folds: 0.92
- Average Recall across 6 folds: 0.90
- Average F1 Score across 6 folds: 0.90
-  [[8.5  0.   0.   0.   0.   0.   0.17 0.17 0.   0.33]
 [0.   8.83 0.   0.   0.   0.   0.   0.   0.33 0.  ]
 [0.17 0.17 7.83 0.   0.   0.83 0.   0.17 0.   0.  ]
 [0.   0.   0.   8.5  0.33 0.   0.   0.33 0.   0.  ]
 [0.   0.   0.17 0.5  8.33 0.17 0.17 0.   0.   0.  ]
 [0.   0.   0.67 0.17 0.   8.17 0.   0.17 0.   0.  ]
 [0.33 0.5  0.17 0.   0.   0.   8.17 0.   0.17 0.  ]
 [0.   0.   0.   0.5  0.   0.17 0.   8.17 0.   0.5 ]
 [0.17 0.   0.5  0.   0.   0.17 0.17 0.   8.   0.33]
 [0.   0.   0.   0.   0.   0.   0.17 0.17 0.17 8.67]]
**sub7**
- Average Accuracy across 6 folds: 91.43%
- Average Precision across 6 folds: 0.93
- Average Recall across 6 folds: 0.91
- Average F1 Score across 6 folds: 0.92
-  [[7.17 0.   0.33 0.   0.17 0.17 0.5  0.   0.   0.  ]
 [0.   7.67 0.17 0.   0.   0.   0.   0.   0.   0.  ]
 [0.33 0.   7.33 0.   0.   0.   0.33 0.   0.   0.  ]
 [0.   0.   0.17 7.33 0.17 0.   0.17 0.33 0.17 0.  ]
 [0.   0.   0.17 0.17 7.33 0.17 0.   0.33 0.   0.  ]
 [0.   0.   0.   0.   0.   7.33 0.5  0.   0.   0.17]
 [0.   0.   0.   0.5  0.   0.17 7.33 0.   0.33 0.  ]
 [0.   0.   0.   0.   0.33 0.33 0.   7.67 0.   0.  ]
 [0.   0.   0.   0.   0.17 0.   0.   0.   8.   0.  ]
 [0.   0.   0.   0.   0.   0.17 0.33 0.17 0.   7.5 ]]
**sub8**
- Average Accuracy across 6 folds: 74.79%
- Average Precision across 6 folds: 0.78
- Average Recall across 6 folds: 0.75
- Average F1 Score across 6 folds: 0.75
-  [[6.17 0.17 0.33 0.17 0.17 0.   0.17 0.33 0.17 0.33]
 [0.5  6.5  0.5  0.17 0.33 0.   0.   0.17 0.   0.  ]
 [0.   0.   7.   0.   0.   0.17 0.5  0.33 0.   0.  ]
 [0.5  0.17 0.33 5.5  0.17 0.17 0.17 0.17 0.17 0.83]
 [0.17 0.17 0.   0.33 6.83 0.33 0.17 0.   0.   0.17]
 [0.33 0.   0.67 0.17 0.33 6.   0.   0.17 0.   0.17]
 [0.17 0.   0.   0.17 0.83 0.   6.   0.17 0.33 0.33]
 [0.   0.   0.   0.5  0.33 0.5  1.17 5.33 0.17 0.  ]
 [0.   0.17 0.17 0.33 0.5  0.   0.17 0.17 4.   0.17]
 [0.67 0.   0.   0.   0.17 0.33 1.17 0.67 0.   5.  ]]
**sub9**
- Average Accuracy across 6 folds: 94.01%
- Average Precision across 6 folds: 0.95
- Average Recall across 6 folds: 0.94
- Average F1 Score across 6 folds: 0.94
-  [[8.33 0.   0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   8.33 0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   7.83 0.   0.   0.33 0.   0.   0.17 0.  ]
 [0.   0.   0.17 7.   0.67 0.   0.   0.17 0.   0.33]
 [0.   0.   0.17 0.17 7.67 0.17 0.   0.17 0.   0.  ]
 [0.   0.   0.33 0.5  0.   7.33 0.   0.   0.   0.17]
 [0.   0.17 0.   0.   0.   0.17 8.33 0.   0.33 0.17]
 [0.   0.   0.   0.   0.17 0.   0.   8.83 0.   0.  ]
 [0.   0.   0.17 0.   0.   0.17 0.17 0.   8.67 0.  ]
 [0.   0.   0.   0.17 0.   0.   0.   0.   0.   8.67]]
**sub10**
- Average Accuracy across 6 folds: 92.89%
- Average Precision across 6 folds: 0.94
- Average Recall across 6 folds: 0.93
- Average F1 Score across 6 folds: 0.93
-  [[8.5  0.   0.17 0.   0.   0.   0.33 0.   0.17 0.  ]
 [0.   8.67 0.   0.   0.17 0.17 0.   0.   0.   0.17]
 [0.17 0.   8.33 0.17 0.   0.17 0.   0.   0.17 0.17]
 [0.   0.   0.   8.5  0.   0.   0.   0.33 0.17 0.  ]
 [0.   0.   0.17 0.   8.17 0.17 0.   0.5  0.   0.17]
 [0.   0.   0.5  0.   0.   8.5  0.   0.   0.   0.17]
 [0.67 0.   0.17 0.   0.   0.   8.33 0.   0.   0.  ]
 [0.   0.   0.   0.   0.17 0.   0.   9.   0.   0.  ]
 [0.33 0.   0.   0.17 0.   0.   0.67 0.   8.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.17 0.   8.83]]

In [31]:
for i in range(1):
    idx = np.arange(len(X_tensor))
    test_idx = idx[sub_marks>8]
    val_idx = idx[(sub_marks>6)&(sub_marks<9)]
    train_idx = idx[sub_marks<7]

    X_train_fold, X_test, X_val_fold = X_tensor[train_idx], X_tensor[test_idx], X_tensor[val_idx]
    y_train_fold, y_test, y_val_fold = y_tensor[train_idx], y_tensor[test_idx], y_tensor[val_idx]

    # Create DataLoader for batching
    train_dataset = TensorDataset(X_train_fold, y_train_fold)
    val_dataset = TensorDataset(X_val_fold, y_val_fold)
    test_dataset = TensorDataset(X_test, y_test)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    # Reset model, criterion, and optimizer for each fold
    model = EMGTransformer(input_dim=input_dim, n_classes=n_classes,
                            n_heads=n_heads, ff_dim=ff_dim, conv_out_dim=32,
                            num_layers=num_layers, dropout=dropout)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # Train the model with early stopping
    train_model(model, train_loader, val_loader,
                criterion, optimizer, epochs=epochs,
                patience=early_stopping_patience)

    # Evaluate the model
    y_pred, y_true = evaluate_model(model, test_loader)

    # Calculate metrics
    fold_accuracy = np.mean(y_pred == y_true)
    fold_precision = precision_score(y_true, y_pred, average='weighted')
    fold_recall = recall_score(y_true, y_pred, average='weighted')
    fold_f1 = f1_score(y_true, y_pred, average='weighted')
    fold_confusion_matrix = confusion_matrix(y_true, y_pred)


    print(f'Accuracy: {fold_accuracy * 100:.2f}%')
    # print('-----------------------')
    # print(f'Precision: {fold_precision:.2f}')
    # print(f'Recall: {fold_recall:.2f}')
    # print(f'F1 Score: {fold_f1:.2f}')
    # print(f'Confusion Matrix:\n{fold_confusion_matrix}')

IndexError: boolean index did not match indexed array along axis 0; size of axis is 548 but size of corresponding boolean axis is 5530